C1: Fix the Dtypes
Fix every dtype problem you found in E1: dates should be datetime, numbers should be numeric.
 Tip: pd.to_datetime() and pd.to_numeric() are your friends. If a numeric conversion fails, look at WHY before reaching for errors='coerce' — know what you're coercing.

In [3]:
import pandas as pd

# Load the original raw dataset
df = pd.read_csv("../data/citibike_weather_daily.csv")

# Convert ride_date from text to datetime
df["ride_date"] = pd.to_datetime(df["ride_date"])

# List the columns that should contain numbers
numeric_cols = [
    "num_rides",
    "avg_duration_min",
    "temp_f",
    "max_temp_f",
    "min_temp_f",
    "wind_speed_knots",
    "precip_in",
    "month"
]

# Confirm each numeric column is stored as a numeric data type
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col])

# Check the data types after cleaning
df.dtypes

ride_date           datetime64[us]
num_rides                    int64
avg_duration_min           float64
temp_f                     float64
max_temp_f                 float64
min_temp_f                 float64
wind_speed_knots           float64
precip_in                  float64
day_of_week                    str
month                        int64
dtype: object

I converted ride_date from text to datetime using pd.to_datetime(). The other columns already had the correct numeric or text data types, so no other dtype changes were needed.

C2: Handle the Coded Missing Values
Deal with whatever your E2 sentinel hunt turned up. First convert any coded values to proper NaN, then decide: drop the row(s), or impute? Justify your choice in a markdown cell — there is more than one defensible answer, but “I didn't notice” is not one of them.
 Tip: Think about how many rows are affected and what imputation would be reasonable for that variable (e.g., a nearby day's value, a median, or zero — which makes sense for THIS variable?).



In [ ]:
import numpy as np

# Change the fake precipitation value to a real missing value
df["precip_in"] = df["precip_in"].replace(99.99, np.nan)

# Check how many missing values are in each column
print(df.isna().sum())

# Remove the one row where precipitation is missing
df = df.dropna().copy()

# Confirm there are no missing values left
print(df.isna().sum())

# Check the new number of rows
print("Rows after cleaning:", df.shape[0])

I changed the value 99.99 in precip_in to NaN because it represents missing data, not real rain. Since there was only one missing row, I dropped it instead of filling in a value. After cleaning, the dataset has 1,609 rows and no missing values.

C3: Encode Day of Week
Your model can't multiply 'Tuesday' by a coefficient. One-hot encode day_of_week into indicator columns.
 Tip: pd.get_dummies(). Look up what drop_first=True does and decide whether to use it — either choice is fine if you can say why.

In [ ]:
# Convert weekday names into 0/1 columns for the model
# drop_first=True keeps one day as a reference day
df = pd.get_dummies(
    df,
    columns=["day_of_week"],
    drop_first=True,
    dtype=int
)

# Check the new columns
display(df.head())
print(df.columns.tolist())

I used pd.get_dummies() to convert day_of_week from text into 0 and 1 columns. I used drop_first=True because one weekday can be used as the reference category. This avoids repeated information and helps prevent multicollinearity in the model.

C4: Build a Trend Feature
Give your model a way to know about the system growth you found in E4. Create either a year column or a days-since-launch index (or both, and pick one for modeling).
 Tip: If ride_date is a proper datetime, .dt.year is one option; subtracting the first date and taking .dt.days is another.

In [6]:
# Find the first date in the cleaned dataset
first_date = df["ride_date"].min()

# Count how many days have passed since the first date
# This gives the model a feature for Citi Bike growth over time
df["days_since_launch"] = (
    df["ride_date"] - first_date
).dt.days

# Check the new feature
display(df[["ride_date", "days_since_launch"]].head())
display(df[["ride_date", "days_since_launch"]].tail())

,ride_date,days_since_launch
0,2013-07-01,0
1,2013-07-02,1
2,2013-07-03,2
3,2013-07-04,3
4,2013-07-05,4


,ride_date,days_since_launch
1605,2018-05-27,1791
1606,2018-05-28,1792
1607,2018-05-29,1793
1608,2018-05-30,1794
1609,2018-05-31,1795


I created a days_since_launch column that starts at 0 on the first date and increases as time passes. The last date has a value of 1795. This helps the model account for the growth of Citi Bike over the years.

C5 (Stretch): Engineer Smarter Features
Optional, for those who want to push the model further. Ideas: a squared temperature term (revisit what you saw in E3 on the hottest days); an is_weekend flag; a rained-at-all binary flag; a US-holidays flag. Each one you add, justify with one sentence tying it to something you observed in EDA.
 Tip: A squared feature is how a LINEAR model captures a CURVED relationship — the model is still linear in its coefficients.

In [7]:
df["temp_f_squared"] = df["temp_f"] ** 2

I created temp_f_squared to help the linear regression model capture a curved relationship between temperature and daily ridership.
image.jpg

C6: Save the Clean Dataset
Save your finished dataframe to data/citibike_weather_daily_clean.csv. This file is what model.ipynb loads — the raw CSV's job is done.

In [ ]:
# Save the cleaned and feature-engineered dataset
df.to_csv("../data/citibike_weather_daily_clean.csv", index=False)